<a href="https://colab.research.google.com/github/pankh-pixel/BBMP-CIVIC-TRACKER/blob/main/civic_tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
import io

In [2]:
url="https://data.opencity.in/dataset/54344a76-a37a-4d05-961c-df9bac5494ad/resource/1342a93b-9a61-4766-9c34-c8357b7926c2/download/b0d6e9ff-5eef-48bf-ba86-985dbe8112d1.csv"

In [3]:
try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    raw_data = pd.read_csv(io.StringIO(response.text))

    print(f"Loaded {len(raw_data)} civic complaints.")
    print("\nFirst 5 rows of raw, messy data:")
    display(raw_data.head())

except Exception as e:
    print(f"Extraction failed: {e}")

Loaded 126974 civic complaints.

First 5 rows of raw, messy data:


,Complaint ID,Category,Sub Category,Grievance Date,Ward Name,Grievance Status,Staff Remarks,Staff Name
0,20771690,Electrical,Street Light Not Working,2025-06-19 10:39:00.000000000,Jagajeevanram Nagar,Registered,1st Assignment Based on Ward Mapping,syed zameer/JE
1,20771689,Electrical,Street Light Not Working,2025-06-19 10:36:00.000000000,Kammanahalli,Registered,1st Assignment Based on Ward Mapping,Vinay Kumar/AE
2,20771688,Solid Waste (Garbage) Related,Garbage dump,2025-06-19 10:35:00.000000000,Banaswadi,Registered,1st Assignment Based on Ward Mapping,Marshal Ward No 27/Marshal
3,20771687,Electrical,Street Light Not Working,2025-06-19 10:34:00.000000000,Gandhi Nagar,Registered,1st Assignment Based on Ward Mapping,syed zameer/JE
4,20771686,Solid Waste (Garbage) Related,Garbage dump,2025-06-19 10:34:00.000000000,Kumaraswamy Layout,Registered,1st Assignment Based on Ward Mapping,Marshal Ward No 181/Marshal


In [18]:
clean_data = raw_data.copy()

In [19]:
print("Columns in Dataset:", clean_data.columns.tolist())

Columns in Dataset: ['Complaint ID', 'Category', 'Sub Category', 'Grievance Date', 'Ward Name', 'Grievance Status', 'Staff Remarks', 'Staff Name']


In [20]:
clean_data = clean_data.dropna(subset=['Category', 'Ward Name'])

In [21]:
clean_data['Category'] = clean_data['Category'].astype(str).str.strip().str.title()
clean_data['Ward Name'] = clean_data['Ward Name'].astype(str).str.strip().str.title()
clean_data['Grievance Status'] = clean_data['Grievance Status'].astype(str).str.strip().str.title()

In [22]:
category_mapping = {
    "Pot Hole": "Pothole",
    "Potholes": "Pothole",
    "Garbage Collection": "Solid Waste Management",
    "Garbage Dump": "Solid Waste Management",
    "Sewage Overflow": "Sanitation & Sewage"
}

In [61]:
clean_data['Category'] = clean_data['Category'].replace(category_mapping)

In [62]:
clean_data['Grievance Date'] = pd.to_datetime(clean_data['Grievance Date'], errors='coerce')

In [63]:
print(f"Cleaning Done. valid records {len(clean_data)} ")
print("\n top 5 most common complaints in Bengaluru:")
display(clean_data['Category'].value_counts().head())

Cleaning Done. valid records 126974 

 top 5 most common complaints in Bengaluru:


,count
Category,
Electrical,42138
Solid Waste (Garbage) Related,38151
Road Maintenance(Engg),14124
Veterinary,7677
Forest,6325


In [64]:
import sqlite3

In [65]:
conn = sqlite3.connect(':memory:')

In [66]:
clean_data.to_sql('Complaints',conn,index=False,if_exists='replace')

126974

In [67]:
print(clean_data['Grievance Status'].value_counts())

Grievance Status
Closed                109410
Registered             10257
Non Relevant            5118
Reopen                  1656
In Progress              359
Long Term Solution       120
Nan                       54
Name: count, dtype: int64


In [68]:
sql_query = """
SELECT
    "Ward Name",
    COUNT(*) as Total_Complaints,
    SUM(CASE WHEN "Grievance Status" = 'Closed' THEN 1 ELSE 0 END) as Complaints_Resolved,
    ROUND((CAST(SUM(CASE WHEN "Grievance Status" = 'Closed' THEN 1 ELSE 0 END) AS FLOAT) / COUNT(*)) * 100, 1) as Resolution_Rate_Percent
FROM complaints
WHERE "Ward Name" != 'Non Ward'
  AND "Ward Name" != 'None'
  AND "Ward Name" IS NOT NULL
GROUP BY "Ward Name"
HAVING Total_Complaints > 100
ORDER BY Resolution_Rate_Percent DESC
LIMIT 10;
"""


In [69]:
most_efficient_wards = pd.read_sql_query(sql_query, conn)
most_efficient_wards.index = most_efficient_wards.index + 1

print("Top 10 Most Efficient Wards in Bengaluru:")
display(most_efficient_wards)

Top 10 Most Efficient Wards in Bengaluru:


,Ward Name,Total_Complaints,Complaints_Resolved,Resolution_Rate_Percent
1,Kempapura Agrahara,185,181,97.8
2,Ejipura,348,331,95.1
3,Chalavadipalya,137,130,94.9
4,Hampi Nagar,598,563,94.1
5,Ramaswamy Palya,197,185,93.9
6,Attiguppe,720,675,93.8
7,Lakkasandra,222,208,93.7
8,Bagalagunte,838,785,93.7
9,Yelachenahalli,1184,1104,93.2
10,Kadugondanahalli,535,498,93.1
